# 第5章　GPUとCUDA、コンテナ運用

**『本格実装 医療診断支援AI（実装編）』のコード**

本文に載っているコードを、章の順にそのまま収めています。紙面のコードは読んで理解するためのもの、こちらは動かすためのものです。

- Python 以外（シェル・YAML・Dockerfile など）は、実行環境が違うので**コードセルにせず、そのまま読める形で置いています**。使う場所を確かめてから実行してください。
- 抜粋である以上、上から順に実行するだけで通るとは限りません。データの取得先やパスは、お手元の環境に合わせてください。
- **教育・研究のためのコードです。患者データをこのノートブックに置かないでください。**

リポジトリ: https://github.com/kewel-corp/book-impl

## 5.2　GPUが本当に使えているか確認する

In [ ]:
import torch
print(torch.__version__)          # PyTorchのバージョン
print(torch.cuda.is_available())  # GPUが使えるか（True なら成功）
print(torch.cuda.get_device_name(0))  # GPUの名前

## CUDAとドライバの「不整合」を読み解く

```text
CUDA driver version is insufficient for CUDA runtime version
```

## nvidia-smi でGPUを見張る

```bash
nvidia-smi        # 一度だけ表示
watch -n 1 nvidia-smi   # 1秒ごとに更新して監視
```

## VRAMの使われ方を見積もる ― 「載るか」を先に計算する

In [ ]:
import torch
torch.cuda.reset_peak_memory_stats()
# ... 1ステップ学習を回す（forward → loss → backward → step）...
peak = torch.cuda.max_memory_allocated() / 1e9
print(f"ピークVRAM: {peak:.2f} GB")   # このバッチ設定で必要な実メモリ

## 手を動かす ― 演習

```bash
mkdir -p ~/work                          # ホスト側にデータ置き場を作る
docker run -it --name medai-test \
  --gpus all \
  --shm-size="8g" \
  -v ~/work:/workspace \
  pytorch/pytorch:2.2.0-cuda12.1-cudnn8-runtime bash
# ↑ここでコンテナの中に入る。以下はコンテナ内で実行
python -c "import torch; print(torch.cuda.get_device_name(0))"  # GPU名が出るか
echo "hello medai" > /workspace/memo.txt   # マウント先にファイルを作る
exit                                        # コンテナから出る
```